***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format


## Setting file paths ---

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Data', 'Census')
    path_config  = os.path.join(path_code, 'config')


## User defined functions ---

exec(open(os.path.join(path_config0, 'Functions.py')).read())


## API key ---

# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()



***

Importing

***

In [ ]:
# Use URL to county fips mapping table
# Import county FIPS codes by state
url_fips = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"
df_fips = pd.read_csv(url_fips, header = None, sep = ',')
df_fips.head()

In [ ]:
# Clean and subset FIPS file
# rename columns
# clean county name
# reformat FIPS field

df_fips.columns = ['State', 'State FIPS', 'County FIPS', 'County Name', 'to be removed']
df_fips = df_fips[['State', 'State FIPS', 'County FIPS', 'County Name']]
df_fips['County Name'] = df_fips['County Name'].str.replace(' County', '')

df_fips['County FIPS'] = df_fips['County FIPS'].astype(str).apply('{:0>3}'.format)
df_fips['State FIPS' ] = df_fips['State FIPS' ].astype(str).apply('{:0>2}'.format)

# show
df_fips.head()

In [ ]:


df_config = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name='CountyFIPS')


df_fips = df_fips.merge(df_config, on=['State', 'State FIPS', 'County FIPS', 'County Name', 'MPO'
                                       , 'MSA_ID', 'MSA_bls', 'MSA_acs', 'Peer MSA', 'Chamber Study']
                        , how='left')
df_fips.head()



***

Exporting

***

In [ ]:


# with pd.ExcelWriter(os.path.join(path_config0, 'Area Codes.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#             df_dist.to_excel(writer, index = False, sheet_name = 'CDcodes')